# PUBLIC 15 — Four-Model OOF Score Generation

- Date: 2026-05-20
- Models: LogisticRegression, GradientBoosting × promo0/promo1
- Params: read from existing final_result.csv (no Optuna rerun)
- Split: StratifiedKFold(n_splits=5, random_state=42)
- Features: 75 features from feature_manifest_used.csv
- Output dir: PUBLIC/results/15_oof_score_or_sensitivity_260520/four_model_oof_scores/

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
PUBLIC = os.path.join(REPO_ROOT, 'PUBLIC')
DATA_DIR = os.path.join(PUBLIC, 'data')
REF_DIR = os.path.join(PUBLIC, 'results', '11_baseline_growth_comparison_260520', 'emergency_four_model_reference')
OUT_DIR = os.path.join(PUBLIC, 'results', '15_oof_score_or_sensitivity_260520', 'four_model_oof_scores')
os.makedirs(OUT_DIR, exist_ok=True)
print('REPO_ROOT:', REPO_ROOT)
print('OUT_DIR:', OUT_DIR)

In [ ]:
CONFIGS = {
    ('LogisticRegression', 'promo0'): {
        'data_csv': os.path.join(DATA_DIR, '06_model_input_promo_0.csv'),
        'manifest_csv': os.path.join(REF_DIR, 'logistic_regression_promo0', 'feature_manifest_used.csv'),
        'params': {'C': 0.1638381730647808, 'class_weight': 'balanced', 'max_iter': 1000, 'random_state': 42},
        'scale': True
    },
    ('LogisticRegression', 'promo1'): {
        'data_csv': os.path.join(DATA_DIR, '06_model_input_promo_1.csv'),
        'manifest_csv': os.path.join(REF_DIR, 'logistic_regression_promo1', 'feature_manifest_used.csv'),
        'params': {'C': 0.0509402675169503, 'class_weight': 'balanced', 'max_iter': 1000, 'random_state': 42},
        'scale': True
    },
    ('GradientBoosting', 'promo0'): {
        'data_csv': os.path.join(DATA_DIR, '06_model_input_promo_0.csv'),
        'manifest_csv': os.path.join(REF_DIR, 'gradient_boosting_promo0', 'feature_manifest_used.csv'),
        'params': {
            'n_estimators': 246, 'learning_rate': 0.0489762903518631,
            'max_depth': 2, 'min_samples_leaf': 189, 'min_samples_split': 473,
            'subsample': 0.7083989692888751, 'max_features': None, 'random_state': 42
        },
        'scale': False
    },
    ('GradientBoosting', 'promo1'): {
        'data_csv': os.path.join(DATA_DIR, '06_model_input_promo_1.csv'),
        'manifest_csv': os.path.join(REF_DIR, 'gradient_boosting_promo1', 'feature_manifest_used.csv'),
        'params': {
            'n_estimators': 219, 'learning_rate': 0.0445518457464106,
            'max_depth': 3, 'min_samples_leaf': 176, 'min_samples_split': 542,
            'subsample': 0.6772458600812739, 'max_features': None, 'random_state': 42
        },
        'scale': False
    }
}
print('Configs loaded:', len(CONFIGS), 'combos')

In [ ]:
def run_oof(model_family, scope, cfg):
    df = pd.read_csv(cfg['data_csv'])
    manifest = pd.read_csv(cfg['manifest_csv'])
    features = manifest[manifest['used_as_feature'] == True]['feature_name'].tolist()
    target = 'is_repurchase'

    X = df[features].values
    y = df[target].values
    n_rows = len(df)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_proba = np.full(n_rows, np.nan)

    fold_records = []
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        if cfg['scale']:
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

        if model_family == 'LogisticRegression':
            model = LogisticRegression(**cfg['params'])
        else:
            model = GradientBoostingClassifier(**cfg['params'])

        model.fit(X_train, y_train)
        val_proba = model.predict_proba(X_val)[:, 1]
        oof_proba[val_idx] = val_proba

        fold_roc = roc_auc_score(y_val, val_proba)
        fold_pr = average_precision_score(y_val, val_proba)
        fold_records.append({
            'model_family': model_family, 'scope': scope,
            'fold': fold_idx, 'val_size': len(val_idx),
            'val_pos': int(y_val.sum()), 'val_neg': int((1 - y_val).sum()),
            'fold_roc_auc': round(fold_roc, 6), 'fold_pr_auc': round(fold_pr, 6)
        })
        print(f'  fold {fold_idx}: roc={fold_roc:.4f} pr={fold_pr:.4f} val_size={len(val_idx)}')

    assert not np.any(np.isnan(oof_proba)), 'NaN in OOF proba'

    oof_roc = roc_auc_score(y, oof_proba)
    oof_pr = average_precision_score(y, oof_proba)
    print(f'  OOF ROC-AUC={oof_roc:.4f}  OOF PR-AUC={oof_pr:.4f}')

    result_df = df[['USER_KEY']].copy() if 'USER_KEY' in df.columns else pd.DataFrame(index=df.index)
    result_df['model_family'] = model_family
    result_df['scope'] = scope
    result_df['is_repurchase'] = y
    result_df['repurchase_score_oof'] = np.round(oof_proba, 8)
    result_df['churn_risk_score_oof'] = np.round(1 - oof_proba, 8)

    metric_rec = {
        'model_family': model_family, 'scope': scope,
        'n_rows': n_rows, 'n_features': len(features),
        'oof_roc_auc': round(oof_roc, 6), 'oof_pr_auc': round(oof_pr, 6),
        'suspicious_high_auc_flag': 'FLAG' if oof_roc >= 0.99 else 'OK',
        'params_json': json.dumps(cfg['params'])
    }

    return result_df, fold_records, metric_rec

In [ ]:
all_oof = []
all_folds = []
all_metrics = []

for (model_family, scope), cfg in CONFIGS.items():
    print(f'Running OOF: {model_family} / {scope}')
    result_df, fold_records, metric_rec = run_oof(model_family, scope, cfg)
    all_oof.append(result_df)
    all_folds.extend(fold_records)
    all_metrics.append(metric_rec)
    print()

print('All OOF runs complete.')

In [ ]:
oof_long = pd.concat(all_oof, ignore_index=True)
oof_long_path = os.path.join(OUT_DIR, '15_oof_score_long.csv')
oof_long.to_csv(oof_long_path, index=False)
print(f'15_oof_score_long.csv saved: {len(oof_long)} rows')

for scope in ['promo0', 'promo1']:
    scope_df = oof_long[oof_long['scope'] == scope].copy()
    lr_sub = scope_df[scope_df['model_family'] == 'LogisticRegression'].reset_index(drop=True)
    gb_sub = scope_df[scope_df['model_family'] == 'GradientBoosting'].reset_index(drop=True)
    assert len(lr_sub) == len(gb_sub), f'row count mismatch: lr={len(lr_sub)} gb={len(gb_sub)}'
    wide = lr_sub[['USER_KEY', 'is_repurchase']].copy() if 'USER_KEY' in lr_sub.columns else lr_sub[['is_repurchase']].copy()
    wide['repurchase_score_oof_logisticregression'] = lr_sub['repurchase_score_oof'].values
    wide['churn_risk_score_oof_logisticregression'] = lr_sub['churn_risk_score_oof'].values
    wide['repurchase_score_oof_gradientboosting'] = gb_sub['repurchase_score_oof'].values
    wide['churn_risk_score_oof_gradientboosting'] = gb_sub['churn_risk_score_oof'].values
    wide_path = os.path.join(OUT_DIR, f'15_oof_score_wide_{scope}.csv')
    wide.to_csv(wide_path, index=False)
    print(f'15_oof_score_wide_{scope}.csv saved: {len(wide)} rows')

In [ ]:
metrics_df = pd.DataFrame(all_metrics)
metrics_path = os.path.join(OUT_DIR, '15_oof_metric_summary.csv')
metrics_df.to_csv(metrics_path, index=False)
print('15_oof_metric_summary.csv:')
print(metrics_df[['model_family', 'scope', 'oof_roc_auc', 'oof_pr_auc', 'suspicious_high_auc_flag']].to_string(index=False))

In [ ]:
folds_df = pd.DataFrame(all_folds)
folds_path = os.path.join(OUT_DIR, '15_oof_fold_distribution_check.csv')
folds_df.to_csv(folds_path, index=False)
print('15_oof_fold_distribution_check.csv saved:', len(folds_df), 'rows')
print(folds_df.to_string(index=False))

In [ ]:
overlap_records = []
for scope in ['promo0', 'promo1']:
    wide_path = os.path.join(OUT_DIR, f'15_oof_score_wide_{scope}.csv')
    wide = pd.read_csv(wide_path)
    lr_col = 'churn_risk_score_oof_logisticregression'
    gb_col = 'churn_risk_score_oof_gradientboosting'
    for threshold in [0.5, 0.6, 0.7]:
        lr_high = wide[lr_col] >= threshold
        gb_high = wide[gb_col] >= threshold
        overlap = (lr_high & gb_high).sum()
        lr_only = (lr_high & ~gb_high).sum()
        gb_only = (~lr_high & gb_high).sum()
        overlap_records.append({
            'scope': scope, 'threshold': threshold,
            'lr_high_risk': int(lr_high.sum()),
            'gb_high_risk': int(gb_high.sum()),
            'overlap_both': int(overlap),
            'lr_only': int(lr_only),
            'gb_only': int(gb_only)
        })
overlap_df = pd.DataFrame(overlap_records)
overlap_path = os.path.join(OUT_DIR, '15_gb_lr_high_risk_overlap.csv')
overlap_df.to_csv(overlap_path, index=False)
print('15_gb_lr_high_risk_overlap.csv saved')
print(overlap_df.to_string(index=False))

In [ ]:
readiness_records = []
for _, row in metrics_df.iterrows():
    roc = row['oof_roc_auc']
    pr = row['oof_pr_auc']
    flag = row['suspicious_high_auc_flag']
    if flag == 'FLAG':
        status = 'BLOCK'
        reason = f'suspicious_high_auc_flag=FLAG (roc={roc})'
    elif roc < 0.50:
        status = 'WARN'
        reason = f'oof_roc_auc={roc} below 0.50 — review before use'
    else:
        status = 'READY'
        reason = f'oof_roc_auc={roc} ok'
    readiness_records.append({
        'model_family': row['model_family'], 'scope': row['scope'],
        'oof_roc_auc': roc, 'oof_pr_auc': pr,
        'suspicious_high_auc_flag': flag,
        'shap_readiness': status, 'segmentation_readiness': status,
        'reason': reason
    })
readiness_df = pd.DataFrame(readiness_records)
readiness_path = os.path.join(OUT_DIR, '15_oof_readiness_for_shap_segmentation.csv')
readiness_df.to_csv(readiness_path, index=False)
print('15_oof_readiness_for_shap_segmentation.csv saved')
print(readiness_df[['model_family','scope','oof_roc_auc','shap_readiness','segmentation_readiness']].to_string(index=False))

In [ ]:
print('=== ALL OUTPUT FILES ===')
for f in sorted(os.listdir(OUT_DIR)):
    fp = os.path.join(OUT_DIR, f)
    print(f'  {f}  ({os.path.getsize(fp)} bytes)')
print('Done.')